# Local T2I-CompBench Official Evaluation For Unclipped SPFC Variants

This notebook evaluates only the two already-generated local unclipped runs `spfc_vfa_unclipped` and `spfc_target_only_uniform_vfa_unclipped`. It validates the existing 100-image run folders under `t2i_compbench_seed13/runs`, pins the official T2I-CompBench checkout, verifies the BLIP VQA and UniDet/Detectron2 local model paths on the available NVIDIA GPUs, then runs the official metrics and report generation for those two methods only.

In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "aim_flow").exists() and (path / "configs" / "t2i_compbench_100_seed13.json").exists():
            return path
    raise RuntimeError("Could not find the aim-flow repo root from the current notebook location.")


REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)

SEED = 13
RUN_SLUG = "t2i_compbench_seed13"
METHODS = ["spfc_vfa_unclipped", "spfc_target_only_uniform_vfa_unclipped"]
EVAL_METHODS = METHODS
CATEGORIES = ["color", "shape", "texture", "spatial"]

MANIFEST_PATH = REPO_DIR / "configs" / "t2i_compbench_100_seed13.json"
RUN_ROOT = REPO_DIR / RUN_SLUG / "runs"
REPORT_SLUG = "local_t2i_compbench_seed13_vfa_unclipped_only"
REPORT_DIR = REPO_DIR / "benchmarks" / "reports" / REPORT_SLUG
EVAL_DIR = REPORT_DIR / "eval"
T2I_REPO_DIR = REPO_DIR / "external" / "T2I-CompBench"
T2I_COMPBENCH_COMMIT = "1b7094991a57f3c22abdd4f6e8ba6c1a15517073"
ENV_PREFIX = REPO_DIR / ".venv" / "t2i-compbench-py310"
EVAL_PYTHON = ENV_PREFIX / "bin" / "python"
CONDA_PKGS_DIR = REPO_DIR / ".conda_pkgs"


def available_nvidia_gpus() -> list[str]:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"nvidia-smi could not detect local GPUs: {result.stderr.strip()}")
    gpu_ids = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    if not gpu_ids:
        raise RuntimeError("No NVIDIA GPUs were detected.")
    return gpu_ids


GPU_IDS = [gpu_id.strip() for gpu_id in os.environ.get("T2I_COMPBENCH_GPUS", "").split(",") if gpu_id.strip()]
if not GPU_IDS:
    GPU_IDS = available_nvidia_gpus()
CUDA_VISIBLE_DEVICES = ",".join(GPU_IDS)
PARALLEL_EVAL_WORKERS = len(GPU_IDS)


def run_cmd(cmd, cwd: Path = REPO_DIR, env: dict | None = None) -> None:
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd))
    merged_env = os.environ.copy()
    if env:
        merged_env.update({key: str(value) for key, value in env.items()})
    subprocess.run(cmd, cwd=str(cwd), env=merged_env, check=True)


def eval_env(extra: dict | None = None) -> dict:
    env = {
        "CUDA_VISIBLE_DEVICES": CUDA_VISIBLE_DEVICES,
        "CUDA_HOME": str(ENV_PREFIX),
        "PATH": f"{ENV_PREFIX / 'bin'}:{os.environ.get('PATH', '')}",
    }
    if extra:
        env.update(extra)
    return env


print("Repo:", REPO_DIR)
print("Manifest:", MANIFEST_PATH)
print("Run root:", RUN_ROOT)
print("Methods:", METHODS)
print("Official repo:", T2I_REPO_DIR)
print("Evaluator Python:", EVAL_PYTHON)
print("Evaluator GPUs:", GPU_IDS)

Repo: /home/cvgl/spfc/aim-flow
Manifest: /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13.json
Run root: /home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs
Methods: ['spfc_vfa_unclipped', 'spfc_target_only_uniform_vfa_unclipped']
Official repo: /home/cvgl/spfc/aim-flow/external/T2I-CompBench
Evaluator Python: /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python
Evaluator GPUs: ['0', '1', '2']


## Validate The Existing Unclipped Runs

This cell checks the two already-generated unclipped method folders under `t2i_compbench_seed13/runs/t2i_compbench` and confirms they contain the full 100-image subset required by the official evaluator.

In [2]:
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

samples = manifest["samples"]
expected_ids = {sample["id"] for sample in samples}
categories = sorted({sample["category"] for sample in samples})
print(f"Manifest subset: {len(samples)} samples across {categories}")

resolved_runs = {method: RUN_ROOT / "t2i_compbench" / method for method in METHODS}
missing_run_dirs = sorted(method for method, path in resolved_runs.items() if not path.exists())
if missing_run_dirs:
    missing_paths = {method: str(resolved_runs[method]) for method in missing_run_dirs}
    raise FileNotFoundError(f"Missing run directories for {missing_run_dirs}: {missing_paths}")
print("Resolved run dirs:", {method: str(path) for method, path in resolved_runs.items()})

validated = {}
for method, method_dir in resolved_runs.items():
    png_paths = sorted(method_dir.glob("*.png"))
    png_ids = {path.stem for path in png_paths}
    missing = sorted(expected_ids - png_ids)
    extra = sorted(png_ids - expected_ids)
    if missing:
        raise RuntimeError(f"{method} is missing {len(missing)} manifest ids, first: {missing[:5]}")
    if extra:
        raise RuntimeError(f"{method} has unexpected ids, first: {extra[:5]}")
    index_path = method_dir / "index.json"
    index_entries = None
    if index_path.exists():
        with index_path.open("r", encoding="utf-8") as f:
            index_data = json.load(f)
        index_entries = len(index_data.get("outputs", []))
    validated[method] = {"png_count": len(png_paths), "index_entries": index_entries}
    print(f"{method}: {len(png_paths)} PNGs, index entries={index_entries}")

if any(item["png_count"] != len(expected_ids) for item in validated.values()):
    raise RuntimeError(f"Run directory count mismatch: {validated}")

Manifest subset: 100 samples across ['color', 'shape', 'spatial', 'texture']
Resolved run dirs: {'spfc_vfa_unclipped': '/home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs/t2i_compbench/spfc_vfa_unclipped', 'spfc_target_only_uniform_vfa_unclipped': '/home/cvgl/spfc/aim-flow/t2i_compbench_seed13/runs/t2i_compbench/spfc_target_only_uniform_vfa_unclipped'}
spfc_vfa_unclipped: 100 PNGs, index entries=50
spfc_target_only_uniform_vfa_unclipped: 100 PNGs, index entries=50


## Pin The Official T2I-CompBench Checkout

In [3]:
if not T2I_REPO_DIR.exists():
    T2I_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "https://github.com/Karine-Huang/T2I-CompBench.git", T2I_REPO_DIR])

run_cmd(["git", "-C", T2I_REPO_DIR, "fetch", "origin", T2I_COMPBENCH_COMMIT])
run_cmd(["git", "-C", T2I_REPO_DIR, "checkout", "--force", T2I_COMPBENCH_COMMIT])

required_scripts = [
    T2I_REPO_DIR / "BLIPvqa_eval" / "BLIP_vqa.py",
    T2I_REPO_DIR / "UniDet_eval" / "2D_spatial_eval.py",
]
for path in required_scripts:
    if not path.exists():
        raise FileNotFoundError(path)
print("Official T2I-CompBench commit:", subprocess.check_output(["git", "-C", str(T2I_REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

$ git -C /home/cvgl/spfc/aim-flow/external/T2I-CompBench fetch origin 1b7094991a57f3c22abdd4f6e8ba6c1a15517073


$ git -C /home/cvgl/spfc/aim-flow/external/T2I-CompBench checkout --force 1b7094991a57f3c22abdd4f6e8ba6c1a15517073
Official T2I-CompBench commit: 1b70949


From https://github.com/Karine-Huang/T2I-CompBench
 * branch            1b7094991a57f3c22abdd4f6e8ba6c1a15517073 -> FETCH_HEAD
HEAD is now at 1b70949 Merge pull request #41 from octo-patch/feature/add-minimax-provider


## Ensure The Local Official Evaluator Environment

The official spatial evaluator depends on Detectron2/UniDet, so this uses a Python 3.10 conda env with PyTorch CUDA 11.7 and a local Detectron2 source build. If the env already imports `torch`, `spacy`, and `detectron2`, this cell skips the setup.

In [4]:
def official_env_imports_ok() -> bool:
    if not EVAL_PYTHON.exists():
        return False
    code = """
import torch, detectron2, spacy
assert torch.cuda.is_available(), 'CUDA is not available to the evaluator env'
spacy.load('en_core_web_sm')
print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))
print('detectron2', getattr(detectron2, '__version__', 'unknown'))
"""
    result = subprocess.run(
        [str(EVAL_PYTHON), "-c", code],
        cwd=str(REPO_DIR),
        env={**os.environ, **eval_env()},
        text=True,
    )
    return result.returncode == 0


if official_env_imports_ok():
    print("Evaluator env is ready; skipping install/build.")
else:
    conda = shutil.which("conda")
    if not conda:
        raise RuntimeError("conda is required to create the local official evaluator env")
    CONDA_PKGS_DIR.mkdir(parents=True, exist_ok=True)
    conda_env = {"CONDA_PKGS_DIRS": str(CONDA_PKGS_DIR)}
    if not EVAL_PYTHON.exists():
        run_cmd([conda, "create", "-y", "-p", ENV_PREFIX, "python=3.10", "pip"], env=conda_env)

    run_cmd([EVAL_PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    run_cmd([EVAL_PYTHON, "-m", "pip", "install", "--no-cache-dir", "torch==2.0.1", "torchvision==0.15.2", "--index-url", "https://download.pytorch.org/whl/cu117"])
    run_cmd([
        EVAL_PYTHON, "-m", "pip", "install", "--no-cache-dir",
        "numpy==1.25.0", "Pillow==9.5.0", "opencv-python-headless==4.7.0.72",
        "spacy==3.5.3", "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.5.0/en_core_web_sm-3.5.0-py3-none-any.whl",
        "accelerate==0.17.0", "fairscale==0.4.4", "timm==0.4.12", "transformers==4.30.2",
        "ruamel.yaml==0.17.32", "pycocotools==2.0.6", "yacs==0.1.8", "fvcore==0.1.5.post20221221",
        "iopath==0.1.9", "omegaconf==2.3.0", "hydra-core==1.3.2", "pandas==2.0.2",
        "tqdm==4.65.0", "ftfy==6.1.1", "regex==2023.6.3", "matplotlib==3.7.1",
        "tabulate==0.9.0", "cloudpickle==2.2.1", "fire==0.5.0", "safetensors==0.3.1",
        "black==22.3.0", "future==0.18.3", "pydot==1.4.2", "tensorboard==2.13.0",
        "diffusers==0.30.3", "pytest==7.4.4", "ninja==1.13.0", "setuptools<70",
    ])
    run_cmd([conda, "install", "-y", "-p", ENV_PREFIX, "gxx_linux-64=11", "gcc_linux-64=11"], env=conda_env)
    run_cmd([
        conda, "install", "-y", "-p", ENV_PREFIX,
        "-c", "nvidia/label/cuda-11.7.1", "-c", "nvidia",
        "cuda-nvcc=11.7.99", "cuda-cccl=11.7.91", "cuda-cudart=11.7.99", "cuda-cudart-dev=11.7.99",
        "cuda-driver-dev=11.7.99", "cuda-nvrtc=11.7.99", "cuda-nvrtc-dev=11.7.99",
        "libcublas=11.10.3.66", "libcublas-dev=11.10.3.66", "libcusparse=11.7.4.91", "libcusparse-dev=11.7.4.91",
        "libcusolver=11.4.0.1", "libcusolver-dev=11.4.0.1", "cuda-version=11.7",
    ], env=conda_env)

    build_env = eval_env({
        "CC": str(ENV_PREFIX / "bin" / "x86_64-conda-linux-gnu-gcc"),
        "CXX": str(ENV_PREFIX / "bin" / "x86_64-conda-linux-gnu-g++"),
        "CPATH": f"{ENV_PREFIX / 'include'}:{os.environ.get('CPATH', '')}",
        "CFLAGS": f"-I{ENV_PREFIX / 'include'}",
        "CXXFLAGS": f"-I{ENV_PREFIX / 'include'}",
        "TORCH_CUDA_ARCH_LIST": "7.5",
        "FORCE_CUDA": "1",
        "MAX_JOBS": "2",
    })
    run_cmd([
        EVAL_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--no-build-isolation", "--no-deps",
        "git+https://github.com/facebookresearch/detectron2.git@5aeb252b194b93dc2879b4ac34bc51a31b5aee13",
    ], env=build_env)
    if not official_env_imports_ok():
        raise RuntimeError("Evaluator env setup completed, but imports still failed.")

torch 2.12.0+cu130 cuda 13.0 NVIDIA GeForce RTX 2080 Ti
detectron2 0.6
Evaluator env is ready; skipping install/build.


## Download UniDet RS200 Weights

In [5]:
UNIDET_WEIGHT = T2I_REPO_DIR / "UniDet_eval" / "experts" / "expert_weights" / "Unified_learned_OCIM_RS200_6x+2x.pth"
UNIDET_WEIGHT_URL = "https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth"
UNIDET_WEIGHT.parent.mkdir(parents=True, exist_ok=True)
if not UNIDET_WEIGHT.exists() or UNIDET_WEIGHT.stat().st_size < 100_000_000:
    tmp = UNIDET_WEIGHT.with_suffix(".pth.tmp")
    if tmp.exists():
        tmp.unlink()
    run_cmd(["curl", "-L", "--fail", "--retry", "3", "--connect-timeout", "30", UNIDET_WEIGHT_URL, "-o", tmp])
    tmp.replace(UNIDET_WEIGHT)
print("UniDet RS200 weight:", UNIDET_WEIGHT, UNIDET_WEIGHT.stat().st_size, "bytes")

UniDet RS200 weight: /home/cvgl/spfc/aim-flow/external/T2I-CompBench/UniDet_eval/experts/expert_weights/Unified_learned_OCIM_RS200_6x+2x.pth 483254220 bytes


## Smoke Test Official BLIP And UniDet Paths

In [8]:
blip_smoke = r'''
import gc
import torch
from models.blip_vqa import blip_vqa
print('cuda available:', torch.cuda.is_available())
model = blip_vqa(
    pretrained='https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth',
    image_size=480,
    vit='base',
    vit_grad_ckpt=False,
    vit_ckpt_layer=0,
)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
print('BLIP loaded on', next(model.parameters()).device)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''
run_cmd([EVAL_PYTHON, "-c", blip_smoke], cwd=T2I_REPO_DIR / "BLIPvqa_eval", env=eval_env())

unidet_smoke = r'''
import gc
import torch
import detectron2
from experts.model_bank import load_expert_model
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('detectron2', getattr(detectron2, '__version__', 'unknown'))
model, transform = load_expert_model(task='obj_detection', ckpt='RS200')
print('UniDet model loaded:', type(model).__name__)
print('Transform:', transform)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''
run_cmd([EVAL_PYTHON, "-c", unidet_smoke], cwd=T2I_REPO_DIR / "UniDet_eval", env=eval_env())

$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python -c 
import gc
import torch
from models.blip_vqa import blip_vqa
print('cuda available:', torch.cuda.is_available())
model = blip_vqa(
    pretrained='https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth',
    image_size=480,
    vit='base',
    vit_grad_ckpt=False,
    vit_ckpt_layer=0,
)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
print('BLIP loaded on', next(model.parameters()).device)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cuda available: True
load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth
BLIP loaded on cuda:0
$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python -c 
import gc
import torch
import detectron2
from experts.model_bank import load_expert_model
print('torch', torch.__version__, 'cuda', torch.cuda.is_availa

/home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/lib/python3.10/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/lib/python3.10/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore
Loading config experts/obj_detection/configs/Base-CRCNN-COCO.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


UniDet model loaded: UnifiedRCNN
Transform: Compose(
    Resize(size=479, interpolation=bilinear, max_size=480, antialias=True)
)


## Run The Official Metrics From Scratch For The Unclipped Variants

This run evaluates only `spfc_vfa_unclipped` and `spfc_target_only_uniform_vfa_unclipped` and writes a fresh official score JSON for those two local methods.

In [ ]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
score_path = EVAL_DIR / "t2i_compbench_scores.json"
if score_path.exists():
    score_path.unlink()
run_cmd([
    EVAL_PYTHON, "scripts/bench_evaluate.py",
    "--benchmark", "t2i_compbench",
    "--manifest", MANIFEST_PATH,
    "--run-root", RUN_ROOT,
    "--methods", *EVAL_METHODS,
    "--output-dir", EVAL_DIR,
    "--t2i-repo-dir", T2I_REPO_DIR,
    "--t2i-categories", *CATEGORIES,
    "--t2i-gpus", *GPU_IDS,
    "--t2i-parallel-workers", str(PARALLEL_EVAL_WORKERS),
    "--execute-official",
], cwd=REPO_DIR, env=eval_env())

with score_path.open("r", encoding="utf-8") as f:
    score_data = json.load(f)
print(json.dumps(score_data["scores"], indent=2, sort_keys=True))

$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13.json --run-root /home/cvgl/spfc/aim-flow/benchmarks/runs/local_t2i_compbench_seed13 --methods spfc rectified_cfgpp cfg base spfc_target_only_uniform --output-dir /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval --t2i-repo-dir /home/cvgl/spfc/aim-flow/external/T2I-CompBench --t2i-categories color shape texture spatial --t2i-gpus 0 1 2 --t2i-parallel-workers 3 --execute-official


  0%|          | 0/8 [00:00<?, ?it/s]

start VQA1/8!
start VQA1/8!
start VQA1/8!


  0%|          | 0/8 [00:00<?, ?it/s]

Number of Processed Images: 25
Creating vqa datasets
Creating model
Number of Processed Images: 25
Creating vqa datasets
Creating model
Number of Processed Images: 25
Creating vqa datasets
Creating model
load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth
Start training
load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth
Start training
load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth
Start training
Generate VQA test result:  [0/1]  eta: 0:00:03    time: 3.4460  data: 1.0611  max mem: 3645
Generate VQA test result: Total time: 0:00:03 (3.5543 s / it)
result file saved to /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval/staged/spfc/shape/examples/annotation1_blip/VQA/result/vqa_result.json
Training time 0:00:03
end VQA1/8!
start VQA2/8!


...

Generate VQA test result:  [0/1]  eta: 0:00:01    time: 1.3736  data: 0.6731  max mem: 3645
Generate VQA test result: Total time: 0:00:01 (1.4942 s / it)
result file saved to /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval/staged/spfc_target_only_uniform/shape/examples/annotation8_blip/VQA/result/vqa_result.json
Training time 0:00:01
end VQA8/8!
BLIP-VQA score: 0.5475080000000001 !



100%|██████████| 8/8 [01:32<00:00, 11.59s/it]
/home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load checkpoint from https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth
Start training
Generate VQA test result:  [0/1]  eta: 0:00:01    time: 1.2228  data: 0.5804  max mem: 3645
Generate VQA test result: Total time: 0:00:01 (1.3922 s / it)
result file saved to /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval/staged/spfc_target_only_uniform/texture/examples/annotation8_blip/VQA/result/vqa_result.json
Training time 0:00:01
end VQA8/8!
BLIP-VQA score: 0.611712 !



100%|██████████| 1/1 [00:14<00:00, 14.08s/it]


vqa result saved in /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval/staged/spfc_target_only_uniform/spatial/examples/labels/annotation_obj_detection_2d
avg score: 0.11054613001509363
eval_scores: /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval/t2i_compbench_scores.json
{
  "base": {
    "color": 0.446252,
    "mean": 0.3727497421875,
    "shape": 0.4820240000000001,
    "spatial": 0.03904296875,
    "texture": 0.52368
  },
  "cfg": {
    "color": 0.8047,
    "mean": 0.5561770063906083,
    "shape": 0.5782399999999999,
    "spatial": 0.2122360255624332,
    "texture": 0.6295320000000001
  },
  "rectified_cfgpp": {
    "color": 0.707324,
    "mean": 0.513110853515625,
    "shape": 0.548608,
    "spatial": 0.1868994140625,
    "texture": 0.6096119999999999
  },
  "spfc": {
    "color": 0.675428,
    "mean": 0.475726451171875,
    "shape": 0.557388,
    "spatial": 0.0785498046875,
    "texture": 0.59154
  },
  "spfc_target_only_unifor

## Build Report Tables And A Qualitative Grid For The Unclipped Variants

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
run_cmd([
    EVAL_PYTHON, "scripts/bench_report.py",
    "--t2i-scores", EVAL_DIR / "t2i_compbench_scores.json",
    "--output-dir", REPORT_DIR,
    "--run-root", RUN_ROOT,
    "--qualitative-manifest", MANIFEST_PATH,
    "--qualitative-output", REPORT_DIR / "qualitative_grid.png",
    "--qualitative-methods", *METHODS,
    "--qualitative-max-prompts", "8",
], cwd=REPO_DIR, env=eval_env())

print("Scores:", EVAL_DIR / "t2i_compbench_scores.json")
print("Report dir:", REPORT_DIR)
print("Qualitative grid:", REPORT_DIR / "qualitative_grid.png")

$ /home/cvgl/spfc/aim-flow/.venv/t2i-compbench-py310/bin/python scripts/bench_report.py --t2i-scores /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/eval/t2i_compbench_scores.json --output-dir /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13 --run-root /home/cvgl/spfc/aim-flow/benchmarks/runs/local_t2i_compbench_seed13 --qualitative-manifest /home/cvgl/spfc/aim-flow/configs/t2i_compbench_100_seed13.json --qualitative-output /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/qualitative_grid.png --qualitative-methods spfc rectified_cfgpp cfg base spfc_target_only_uniform --qualitative-max-prompts 8
report_outputs:
  t2i_csv: /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/t2i_compbench_table.csv
  t2i_md: /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/t2i_compbench_table.md
  qualitative_grid: /home/cvgl/spfc/aim-flow/benchmarks/reports/local_t2i_compbench_seed13/qualitative_g